# 3D Slicer + TCIA kidney CT + DICOM SEG on Google Colab

A worked example of viewing TCIA open-access data — the **C4KC-KiTS** kidney CT with its **DICOM SEG**
segmentation (kidney + tumor) — in the **full 3D Slicer application**, running on a Colab CPU backend and
streamed into the cell below.

This mirrors the CT+SEG example in
[TCIA_Segmentations.ipynb](https://github.com/kirbyju/TCIA_Notebooks/blob/main/TCIA_Segmentations.ipynb),
but instead of a lightweight viewer it runs **all of Slicer**: it installs the QuantitativeReporting
extension to load DICOM SEG natively, shows the CT with the segmentation overlaid in the slice views,
and builds a **3D surface** of the segments. Use a **Chrome** browser; a plain **CPU runtime** is fine.


## 1. Install dependencies (~1 min)


In [ ]:
%%bash
set -e
cd /content
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq --no-install-recommends \
  xvfb xclip matchbox-window-manager fonts-dejavu-core libgl1-mesa-dri libglu1-mesa \
  gstreamer1.0-plugins-base gstreamer1.0-plugins-good gstreamer1.0-plugins-bad gstreamer1.0-plugins-ugly \
  python3-gi gir1.2-gstreamer-1.0 gir1.2-gst-plugins-base-1.0 python3-xlib \
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-randr0 libxcb-render-util0 libxcb-shape0 \
  libxcb-sync1 libxcb-xfixes0 libxcb-xinerama0 libxcb-xkb1 libxkbcommon-x11-0 libxcb-cursor0 libxcb-util1 \
  libodbc2 libpq5 libpulse-mainloop-glib0 libpcre2-16-0 \
  libxcomposite1 libxdamage1 libxtst6 libhwloc15 libnspr4 libnss3 >/dev/null
apt-get install -y -qq --no-install-recommends libasound2 libcups2 >/dev/null 2>&1 \
  || apt-get install -y -qq --no-install-recommends libasound2t64 libcups2t64 >/dev/null
pip install -q websockets aioquic
echo 'deps installed'

## 2. Get Slicer + install QuantitativeReporting (DICOM SEG support)

Installs the QuantitativeReporting extension **into the Slicer install now** (headless), so the launch
below is restart-free and can load DICOM SEG natively. Adds ~1 min.


In [ ]:
%%bash
set -e
cd /content
REPO=${DESKTOPIA_REPO:-https://github.com/pieper/desktopia}
BRANCH=${DESKTOPIA_BRANCH:-software-render}
rm -rf /content/desktopia
git clone -q --branch "$BRANCH" "$REPO" /content/desktopia || git clone -q "$REPO" /content/desktopia
if ! ls -d /opt/Slicer-*/ >/dev/null 2>&1; then
  echo 'downloading 3D Slicer (~400 MB)...'
  curl -L --retry 3 'https://download.slicer.org/download?os=linux&stability=release' | tar -xz -C /opt
fi
SDIR=$(ls -d /opt/Slicer-*/ | head -1); echo "Slicer: $SDIR"
cat > /tmp/install_qr.py <<'PY'
import slicer
emm = slicer.app.extensionsManagerModel()
emm.interactive = False
try: emm.updateExtensionsMetadataFromServer(True, True)
except Exception as e: print('metadata:', e, flush=True)
ok = False
try:
    emm.downloadAndInstallExtensionByName('QuantitativeReporting', True, True)
    ok = emm.isExtensionInstalled('QuantitativeReporting')
except Exception as e:
    print('install error:', e, flush=True)
print('QR_INSTALLED=', ok, flush=True)
slicer.util.exit(0 if ok else 1)
PY
echo 'installing QuantitativeReporting...'
LIBGL_ALWAYS_SOFTWARE=1 GALLIUM_DRIVER=llvmpipe HOME=/root \
  xvfb-run -a "$SDIR/Slicer" --no-splash --no-main-window --ignore-slicerrc \
  --python-script /tmp/install_qr.py || echo 'QR install returned nonzero (see log above)'

## 3. Download the TCIA kidney CT + DICOM SEG

Direct download by Series UID from TCIA (no index to fetch first). C4KC-KiTS patient **KiTS-00000**: a
~600-slice CT and its DICOM SEG. ~1 min.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'tcia_utils'], check=True)
from tcia_utils import nbia
CT  = '1.3.6.1.4.1.14519.5.2.1.6919.4624.113493579075669637574394466994'   # CT (reference series)
SEG = '1.2.276.0.7230010.3.1.3.0.74366.1588583084.764537'                  # DICOM SEG (kidney + tumor)
nbia.downloadSeries([CT, SEG], input_type='list', path='/content/tciaDownload')
print('download complete -> /content/tciaDownload')

## 4. Launch Slicer — loads the CT + SEG and builds the 3D surface

Slicer imports the DICOM, overlays the segmentation on the CT slices, and renders a 3D surface of the
segments (low-poly, fast even on CPU). Click into the frame to interact.


In [ ]:
import os, time, glob, pathlib, subprocess
os.chdir('/content'); WORK = '/content/desktopia'
WIDTH, HEIGHT, FPS, BITRATE = 1280, 720, 15, 4000

# STARTUP runs inside Slicer: load /content/tciaDownload, overlay SEG, build 3D surfaces, frame views.
STARTUP = r'''
import slicer
from DICOMLib import DICOMUtils
loaded = []
with DICOMUtils.TemporaryDICOMDatabase() as db:
    DICOMUtils.importDicom('/content/tciaDownload', db)
    for p in db.patients():
        loaded += DICOMUtils.loadPatientByUID(p)
print('loaded nodes:', loaded, flush=True)
for seg in slicer.util.getNodesByClass('vtkMRMLSegmentationNode'):
    seg.CreateClosedSurfaceRepresentation()           # 3D surface of the segments
    dn = seg.GetDisplayNode()
    if dn:
        dn.SetVisibility(True); dn.SetVisibility3D(True)
slicer.app.layoutManager().setLayout(slicer.vtkMRMLLayoutNode.SlicerLayoutFourUpView)
vols = slicer.util.getNodesByClass('vtkMRMLScalarVolumeNode')
if vols:
    slicer.util.setSliceViewerLayers(background=vols[0], fit=True)
slicer.util.resetSliceViews()
try:
    tdv = slicer.app.layoutManager().threeDWidget(0).threeDView()
    tdv.resetFocalPoint(); tdv.resetCamera()
except Exception as e:
    print('3d reset:', e, flush=True)
'''

env = dict(os.environ, DISPLAY=':2', LIBGL_ALWAYS_SOFTWARE='1', GALLIUM_DRIVER='llvmpipe', HOME='/root')
for pat in ('server.py', 'SlicerApp-real'):
    subprocess.run(['pkill', '-f', pat], check=False)
for proc in ('Xvfb', 'matchbox-window-manager'):
    subprocess.run(['pkill', '-x', proc], check=False)
time.sleep(1)

subprocess.run('openssl req -x509 -newkey ec -pkeyopt ec_paramgen_curve:prime256v1 '
               '-keyout /tmp/k.pem -out /tmp/c.pem -days 1 -nodes -subj /CN=desktopia',
               shell=True, check=True, stderr=subprocess.DEVNULL)
subprocess.Popen(f'Xvfb :2 -screen 0 {WIDTH}x{HEIGHT}x24 +extension GLX +render -noreset',
                 shell=True, env=env, stdout=open('/tmp/xvfb.log','w'), stderr=subprocess.STDOUT)
for _ in range(80):
    if os.path.exists('/tmp/.X11-unix/X2'): break
    time.sleep(0.25)
subprocess.Popen('matchbox-window-manager -use_titlebar no', shell=True, env=env,
                 stdout=open('/tmp/wm.log','w'), stderr=subprocess.STDOUT)
time.sleep(1)

SDIR = sorted(glob.glob('/opt/Slicer-*/'))[0]
pathlib.Path('/tmp/slicer_startup.py').write_text(STARTUP)
subprocess.Popen(f'{SDIR}/Slicer --no-splash --python-script /tmp/slicer_startup.py',
                 shell=True, env=env, stdout=open('/tmp/slicer.log','w'), stderr=subprocess.STDOUT)

pathlib.Path(f'{WORK}/client/status.json').write_text('{"ready":true,"transport":"websocket"}')
subprocess.Popen('python3 server.py --cert /tmp/c.pem --key /tmp/k.pem '
                 f'--source xvfb --width {WIDTH} --height {HEIGHT} --fps {FPS} --bitrate {BITRATE} '
                 '--ws-plain --serve-dir client',
                 shell=True, env=env, cwd=WORK,
                 stdout=open('/tmp/server.log','w'), stderr=subprocess.STDOUT)
time.sleep(8)
print('--- server.log ---'); print(open('/tmp/server.log').read()[-1200:])
print('Slicer is loading the CT+SEG; the view appears below in a few seconds.')

## 5. Open Slicer in this cell

**Give it a minute.** After you run this cell it takes ~a minute for Slicer to start, load the CT +
segmentation, and build the 3D surface — the frame may sit on a connecting/loading screen until then.
Please be patient until the **rendered kidney appears** in the views. Once it does, you can use Slicer
normally (mouse + keyboard) right in the frame.


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(4434, path='/index.html', height=600, cache_in_notebook=False)

### Notes / troubleshooting

- **Slicer's log:** `print(open('/tmp/slicer.log').read())` — shows the loaded nodes and any SEG error.
- **No segmentation shown:** confirm cell 2 printed `QR_INSTALLED= True`. If not, re-run cell 2.
- **Frame stuck connecting:** open in a tab — `from google.colab import output; output.serve_kernel_port_as_window(4434, path='/index.html')`.
